# AlphaFold Confidence Score Analysis: Human p53 (TP53)

This notebook fetches AlphaFold's precomputed prediction for human **p53** (UniProt `P04637`) from the [AlphaFold Protein Structure Database](https://alphafold.ebi.ac.uk/) and explores its confidence metrics:

- **pLDDT** (per-residue confidence, 0-100) — how confident AlphaFold is about the *local* structure at each residue
- **PAE** (Predicted Aligned Error) — how confident AlphaFold is about the *relative position/orientation* between every pair of residues

p53 is a great case study because it has both a well-folded DNA-binding domain (high confidence) and long intrinsically disordered regions (low confidence) — so we should see that contrast clearly in the data.

**Sections:**
1. Fetch data from the AlphaFold DB API
2. Parse pLDDT per-residue scores
3. Statistical summary + confidence intervals on pLDDT
4. Visualize pLDDT along the sequence, with confidence bands
5. Visualize the PAE matrix as a heatmap
6. 3D structure colored by pLDDT

In [ ]:
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

UNIPROT_ID = "P04637"  # Human TP53 / p53

api_url = f"https://alphafold.ebi.ac.uk/api/prediction/{UNIPROT_ID}"
resp = requests.get(api_url, timeout=30)
resp.raise_for_status()
entry = resp.json()[0]

print("Gene:", entry["gene"])
print("Description:", entry["uniprotDescription"])
print("Sequence length:", entry["sequenceEnd"])
print("Model version:", entry["latestVersion"])
print("Global pLDDT (mean):", entry["globalMetricValue"])
print()
print("Fraction very low (<50):", entry["fractionPlddtVeryLow"])
print("Fraction low (50-70):", entry["fractionPlddtLow"])
print("Fraction confident (70-90):", entry["fractionPlddtConfident"])
print("Fraction very high (>90):", entry["fractionPlddtVeryHigh"])

In [ ]:
# Download the per-residue confidence (pLDDT) JSON and the PAE JSON
plddt_doc = requests.get(entry["plddtDocUrl"], timeout=30).json()
pae_doc = requests.get(entry["paeDocUrl"], timeout=30).json()

# plddt_doc is a dict with parallel residueNumber / confidenceScore / confidenceCategory arrays
plddt_scores = np.array(plddt_doc["confidenceScore"])
residue_numbers = np.arange(1, len(plddt_scores) + 1)

df = pd.DataFrame({"residue": residue_numbers, "pLDDT": plddt_scores, "aa": list(entry["sequence"])})
df.head()

## Statistical summary and confidence intervals

We compute the overall distribution of pLDDT scores, then bootstrap a 95% confidence interval on the **mean pLDDT**, and separately look at the well-known structural domains of p53 to see how confidence differs by region:

- **Transactivation domain**: residues 1-61 (disordered)
- **DNA-binding domain**: residues 94-312 (well-folded)
- **Tetramerization domain**: residues 323-356 (folded)
- **C-terminal regulatory domain**: residues 363-393 (disordered)

In [ ]:
print(df["pLDDT"].describe())
print()

# Bootstrap 95% CI on the mean pLDDT
rng = np.random.default_rng(42)
boot_means = np.array([
    rng.choice(df["pLDDT"].values, size=len(df), replace=True).mean()
    for _ in range(10_000)
])
ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5])
print(f"Mean pLDDT: {df['pLDDT'].mean():.2f}")
print(f"95% bootstrap CI on the mean: [{ci_low:.2f}, {ci_high:.2f}]")

# Standard-error-based CI for comparison (assumes near-normal sampling distribution of the mean)
sem = stats.sem(df["pLDDT"])
t_ci = stats.t.interval(0.95, len(df) - 1, loc=df["pLDDT"].mean(), scale=sem)
print(f"95% t-distribution CI on the mean:  [{t_ci[0]:.2f}, {t_ci[1]:.2f}]")

In [ ]:
domains = {
    "Transactivation (1-61)": (1, 61),
    "DNA-binding (94-312)": (94, 312),
    "Tetramerization (323-356)": (323, 356),
    "C-terminal reg. (363-393)": (363, 393),
}

rows = []
for name, (start, end) in domains.items():
    mask = (df["residue"] >= start) & (df["residue"] <= end)
    sub = df.loc[mask, "pLDDT"]
    sem = stats.sem(sub)
    ci = stats.t.interval(0.95, len(sub) - 1, loc=sub.mean(), scale=sem)
    rows.append({
        "domain": name,
        "n_residues": len(sub),
        "mean_pLDDT": round(sub.mean(), 2),
        "std": round(sub.std(), 2),
        "95%_CI_low": round(ci[0], 2),
        "95%_CI_high": round(ci[1], 2),
    })

domain_df = pd.DataFrame(rows)
domain_df

## pLDDT along the sequence

AlphaFold's standard confidence bands: **very high** (>90, dark blue), **confident** (70-90, light blue), **low** (50-70, yellow), **very low** (<50, orange).

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

bands = [(90, 100, "#0053D6", "Very high (>90)"),
         (70, 90, "#65CBF3", "Confident (70-90)"),
         (50, 70, "#FFDB13", "Low (50-70)"),
         (0, 50, "#FF7D45", "Very low (<50)")]
for low, high, color, label in bands:
    ax.axhspan(low, high, color=color, alpha=0.15, label=label)

ax.plot(df["residue"], df["pLDDT"], color="black", linewidth=0.8)

for name, (start, end) in domains.items():
    ax.axvspan(start, end, color="gray", alpha=0.08)
    ax.text((start + end) / 2, 102, name.split(" (")[0], ha="center", fontsize=8, rotation=0)

ax.axhline(df["pLDDT"].mean(), color="red", linestyle="--", linewidth=1, label=f"Mean = {df['pLDDT'].mean():.1f}")
ax.fill_between(df["residue"], ci_low, ci_high, color="red", alpha=0.1)

ax.set_xlabel("Residue number")
ax.set_ylabel("pLDDT confidence score")
ax.set_title(f"Per-residue pLDDT confidence — {entry['gene']} / {entry['uniprotId']}")
ax.set_ylim(0, 108)
ax.legend(loc="lower right", fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

## Predicted Aligned Error (PAE) heatmap

PAE captures pairwise confidence: for residue pair (i, j), how much error is expected in the position of residue *j* if the structure were aligned on residue *i*. Dark/blue squares along the diagonal indicate well-defined domains; bright off-diagonal regions show where relative domain orientation is uncertain (common for multi-domain or disordered proteins).

In [ ]:
pae_matrix = np.array(pae_doc[0]["predicted_aligned_error"])
print("PAE matrix shape:", pae_matrix.shape)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(pae_matrix, cmap="Greens_r", vmin=0, vmax=pae_doc[0].get("max_predicted_aligned_error", 31.75))
for name, (start, end) in domains.items():
    ax.axvline(start, color="gray", linewidth=0.5, alpha=0.5)
    ax.axhline(start, color="gray", linewidth=0.5, alpha=0.5)
ax.set_xlabel("Residue j")
ax.set_ylabel("Residue i")
ax.set_title(f"Predicted Aligned Error — {entry['gene']}")
fig.colorbar(im, ax=ax, label="Expected position error (Å)")
plt.tight_layout()
plt.show()